Extract NOAA CPD2 tarballs and save data to parquet files.
Provide plotting functions for data visualization.
Evaluate manual zero and span checks of nephelometer from October 2023.

joerg.klausen@meteoswiss.ch

In [1]:
import os
# import polars as pl
# import pandas as pd
# import matplotlib.pyplot as plt
# from io import BytesIO
# import json
# import re
# import tarfile
# import zipfile
from dataprocessor.cpd2 import CPD2
from dataprocessor.ae33 import AE33


Matplotlib created a temporary config/cache directory at /tmp/matplotlib-784sd92t because the default path (/prod/zue/fc_development/users/atr/cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [4]:
# process CPD2 data files
cpd2 = CPD2()
years = ["2023"]
for year in years:
    source = os.path.join("/product_data/data/pay/Kenya/MKN/incoming/aerosol", year)
    target = os.path.join("results", "aerosol", year)
    errors = cpd2.tarballs_to_parquet(source=source, target=target)
    print(errors)
print("done")

CPD2 initialized.
Processing source /product_data/data/pay/Kenya/MKN/incoming/aerosol/2023 ...
Processing mkn_20230704T060006Z.tar.gz ...
Processing A11_20230704T050258Z ...
Processing S11_20230704T050001Z ...
Processing mkn_20230704T040006Z.tar.gz ...
Processing A11_20230704T030258Z ...
Processing S11_20230704T030002Z ...
Processing mkn_20230704T050006Z.tar.gz ...
Processing S11_20230704T040002Z ...
Processing A11_20230704T040559Z ...
Processing mkn_20230704T030006Z.tar.gz ...
Processing A11_20230704T020259Z ...
Processing S11_20230704T020001Z ...
Processing mkn_20230704T010006Z.tar.gz ...
Processing S11_20230704T000002Z ...
Processing A11_20230704T000259Z ...
Processing mkn_20230704T020006Z.tar.gz ...
Processing A11_20230704T010259Z ...
Processing S11_20230704T010003Z ...
Processing mkn_20230704T000006Z.tar.gz ...
Processing A11_20230703T230259Z ...
Processing S11_20230704T000000Z ...
Processing S11_20230703T230001Z ...
Processing mkn_20230704T220006Z.tar.gz ...
Processing S11_202307

In [ ]:
# process AE33 data files
ae33 = AE33()
years = ["2023"]
for year in years:
    source = os.path.join("/product_data/data/pay/Kenya/MKN/incoming/ae33", year)
    target = os.path.join("results", "ae33", year)
    errors = ae33.tarballs_to_parquet(source=source, target=target)
    print(errors)
print("done")

In [ ]:
df = pl.read_parquet("/home/zue/users/jkl/Public/git/gawkenya/results/ae33/2023/ae33.parquet")

cols = ["DateTime", "BC1", "BC2", "BC3", "BC4", "BC5", "BC6", "BC7"]
display(df.select(cols).describe())

In [ ]:

df_flagged = ae33.flag_spurious_data(df)

cols_flags = ["DateTime", "flags_BC1", "flags_BC2", "flags_BC3", "flags_BC4", "flags_BC5", "flags_BC6", "flags_BC7"]
display(df_flagged.select(cols_flags).describe())

In [ ]:

# df, cutoffs = ae33.remove_extremes(df, q=0.00001)
# display(cutoffs)
# Filter the DataFrame to include only rows where 'spurious_flag' is False
# filtered_df = df.filter(df['flags'] == False)

ae33.plot_aethalometer_data(df, start="2023-10-01", end="2023-10-31")

In [ ]:
# df["flags_BC1"].describe()
ae33.plot_aethalometer_data(df, start="2023-07-01", end="2023-07-31")

In [ ]:
# import polars as pl

# def flag_spurious_data(dataframe, value_threshold=0, consecutive_threshold=2):
#     """
#     Flag spurious data in a polars DataFrame consisting of time series data.

#     Parameters:
#     - dataframe: polars DataFrame
#     - value_threshold: threshold for considering values around zero (default: 0)
#     - consecutive_threshold: threshold for consecutive occurrences (default: 2)

#     Returns:
#     - polars DataFrame with an additional 'spurious_flag' column
#     """
#     spurious_flags = []

#     for column in dataframe.columns:
#         # Identify spurious data based on the specified thresholds
#         spurious_mask = (
#             (dataframe[column] <= value_threshold) & 
#             (dataframe[column].shift(-1) > value_threshold) & 
#             (dataframe[column].shift(consecutive_threshold) > value_threshold)
#         ) | (
#             (dataframe[column] <= value_threshold) & 
#             (dataframe[column].shift(1) > value_threshold) & 
#             (dataframe[column].shift(-consecutive_threshold) > value_threshold)
#         )

#         spurious_flags.append(spurious_mask)

#     # Create a new column 'spurious_flag' in the DataFrame
#     dataframe = dataframe.with_column('spurious_flag', pl.col(spurious_flags).any())

#     return dataframe

# # Example usage:
# # Assuming 'your_time_series_data.csv' is your input CSV file
# input_file = 'your_time_series_data.csv'

# # Read the CSV file into a polars DataFrame
# df = pl.read_csv(input_file)

# # Call the function to flag spurious data
# df_with_spurious_flags = flag_spurious_data(df)

# # Display the resulting DataFrame with spurious flags
# print(df_with_spurious_flags)


In [ ]:
# # Concatenate the individual DataFrames into a single one
# combined_data = pl.concat(dataframes)

# # Assuming you have datetime columns, replace 'datetime_column' with the actual column name.
# datetime_column = 'timestamp'

# # Perform aggregation by datetime_column
# agg_data = (
#     combined_data
#     .with_column(combined_data[datetime_column].cast(pl.Date32))
#     .groupby(datetime_column)
#     .agg(pl.sum(combined_data['value_column']))
#     .sort(datetime_column)
# )